# DockingCopilot Skill 可复现演示

本 notebook 从头顺序执行一次完整的分子对接闭环（真实运行，非模拟）。

**科学前提**：AutoDock Vina 的 affinity 是经验打分函数与构象搜索的结果，
不等于实验结合自由能、Kd、Ki、IC50 或任何药效结论。

## 1. 环境和版本

In [ ]:
import json
import platform
import sys

import Bio
import meeko
import rdkit

print("python   ", sys.version.split()[0])
print("rdkit    ", rdkit.__version__)
print("meeko    ", meeko.__version__)
print("biopython", Bio.__version__)
try:
    from vina import Vina

    print("vina     ", Vina.version)
except Exception as exc:  # noqa: BLE001
    print("vina python api unavailable:", exc)
print("platform ", platform.platform(), platform.machine())

## 2. 加载仓库示例输入（相对路径）

In [ ]:
from pathlib import Path

# notebook 位于 notebooks/，项目根为其上一级；不使用任何绝对路径
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "backend"))

input_path = PROJECT_ROOT / "examples" / "inputs" / "multiple_ligands.json"
payload = json.loads(input_path.read_text(encoding="utf-8"))
print("job_name:", payload["job_name"])
print(json.dumps(payload["ligands"], indent=2))

## 3. 校验输入（Pydantic + jsonschema 双重校验）

In [ ]:
import jsonschema

from app.schemas import DockingInput

docking_input = DockingInput.model_validate(payload)
schema = json.loads(
    (PROJECT_ROOT / "schemas" / "docking_input.schema.json").read_text(encoding="utf-8")
)
jsonschema.validate(instance=payload, schema=schema)
print(
    "Pydantic OK, jsonschema OK;",
    f"{len(docking_input.ligands)} ligands,",
    f"seed={docking_input.docking_parameters.seed}",
)

## 4. 准备 ligand（SMILES → 3D 构象 → 描述符 → PDBQT）

构象经 ETKDGv3（seed=42）生成、MMFF94 优化（不可用时受控回退 UFF 并记录警告），
PDBQT 经 Meeko 写出。

In [ ]:
import tempfile

from IPython.display import SVG, display

from app.services import ligand_service

with tempfile.TemporaryDirectory() as tmp:
    tmp_dir = Path(tmp)
    demo_result = ligand_service.prepare_ligand(
        docking_input.ligands[0], docking_input.ligand_preparation, tmp_dir
    )
    meta = json.loads((tmp_dir / "ligand_metadata.json").read_text(encoding="utf-8"))
    print("ligand_id:", demo_result.ligand_id, "| status:", demo_result.status)
    print("force field:", meta["force_field_used"])
    print(
        "conformer energy:", round(meta["conformer_energy"], 3),
        "kcal/mol (力场能量，不是结合能)",
    )
    print("preparation warnings:", meta["preparation_warnings"])
    display(SVG((tmp_dir / "ligand_2d.svg").read_bytes()))

## 5. 分析 receptor（链/原子/异源实体/包围盒）

In [ ]:
from app import config
from app.services import receptor_service
from app.utils.files import safe_resolve

receptor_path = safe_resolve(docking_input.receptor.source, config.PROJECT_ROOT)
report = receptor_service.analyze_receptor(receptor_path)
print("receptor:", docking_input.receptor.source)
print("chains:", report["chains"])
print("atom_count:", report["atom_count"], "| residues:", report["residue_count"])
print("hetero entities:", len(report["hetero_entities"]))
bbox_min, bbox_max = receptor_service.bounding_box_from_report(report)
print("bounding box min:", [round(x, 2) for x in bbox_min], "Å")
print("bounding box max:", [round(x, 2) for x in bbox_max], "Å")

## 6. 显示 docking box（四模式之一：explicit）

In [ ]:
from app.services import box_service

box_report = dict(report)
box_report["bounding_box"] = (bbox_min, bbox_max)
box_result = box_service.build_box(docking_input.docking_box, box_report)
print("mode:  ", box_result.mode)
print("center:", box_result.center, "Å")
print("size:  ", box_result.size, "Å")
print("volume:", box_result.volume_angstrom3, "Å³")
print("warnings:", box_result.warnings)

## 7. 运行 Vina（真实对接，固定 seed=42）

调用完整工作流 `run_job`：validate → ligand → receptor → box → vina →
parse → metrics → interpret → manifest → package。

In [ ]:
from app.workflow.runner import run_job

outcome = run_job(docking_input)
print("job_id:", outcome.job_id)
print("state: ", outcome.state.value)
assert outcome.state.value == "SUCCESS", outcome.structured_error
job_dir = outcome.job_dir
for res in outcome.output.results:
    print(
        f"{res.ligand_id}: status={res.status} "
        f"best={res.best_affinity_kcal_mol} kcal/mol "
        f"poses={len(res.poses)}"
    )

## 8. 解析 top poses

In [ ]:
print(f"{'ligand':<14}{'rank':>5}{'affinity':>12}{'rmsd_lb':>9}{'rmsd_ub':>9}")
for res in outcome.output.results:
    for pose in res.poses[:3]:  # 只展示 top-3，避免输出过大
        print(
            f"{res.ligand_id:<14}{pose.rank:>5}"
            f"{pose.affinity_kcal_mol:>12.3f}{pose.rmsd_lb:>9.3f}{pose.rmsd_ub:>9.3f}"
        )

## 9. 计算描述符

In [ ]:
print(f"{'ligand':<14}{'MolWt':>9}{'LogP':>8}{'TPSA':>9}{'HBD':>5}{'HBA':>5}{'rotB':>6}{'heavy':>7}")
for res in outcome.output.results:
    d = res.descriptors
    print(
        f"{res.ligand_id:<14}{d['MolWt']:>9.2f}{d['LogP']:>8.2f}{d['TPSA']:>9.2f}"
        f"{d['HBD']:>5}{d['HBA']:>5}{d['rotatable_bonds']:>6}{d['heavy_atom_count']:>7}"
    )

## 10. 生成比较数据（批次内展示性相对指标）

只有同受体、同准备流程、同 box、同参数才可相对比较；公式透明记录在
`normalization_method` 字段。

In [ ]:
from app.schemas import ComparisonContext
from app.services import comparison_service
from app.utils.hashing import sha256_file

context = ComparisonContext(
    receptor_sha256=sha256_file(receptor_path),
    box_mode=box_result.mode,
    box_center=box_result.center,
    box_size=box_result.size,
    docking_parameters=docking_input.docking_parameters.model_dump(),
)
comparison = comparison_service.compare_in_batch(
    outcome.output.results, context=context, job_id=outcome.job_id
)
print("normalization_method:")
print(comparison.normalization_method)
print()
for ligand_id, entry in comparison.metrics.items():
    print(ligand_id, "->", {k: v for k, v in entry.items() if k != "status"})

## 11. 2D 图、3D 文件路径与雷达图

雷达图标题固定为 **Relative Docking & Developability Profile**（展示性指标，
不是真实结合能力或药效评分）。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 2D 图与 3D pose 文件路径（来自真实任务产物）
for res in outcome.output.results:
    svg_path = job_dir / "ligands" / res.ligand_id / "ligand_2d.svg"
    if svg_path.is_file():
        print(res.ligand_id, "2D:", svg_path.relative_to(job_dir))
        display(SVG(svg_path.read_bytes()))
    for pose in res.poses[:2]:
        print(" ", "pose:", pose.pose_file, "| pdb:", pose.pose_pdb_file)

# 雷达图：统一方向归一化后的展示性指标
dimensions = [
    ("Relative Affinity", "relative_affinity"),
    ("Ligand Efficiency", "ligand_efficiency_rank_norm"),
    ("Pose Consistency", "pose_consistency"),
    ("Size Fit", "molwt_fit"),
    ("LogP Fit", "logp_fit"),
    ("TPSA Fit", "tpsa_fit"),
    ("Flexibility Fit", "flexibility_fit"),
    ("Prep Quality", "preparation_quality"),
]
labels = [label for label, _ in dimensions]
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"polar": True})
for res in outcome.output.results:
    entry = comparison.metrics[res.ligand_id]
    values = [float(entry.get(key, 0.0) or 0.0) for _, key in dimensions]
    values += values[:1]
    ax.plot(angles, values, label=res.ligand_id)
    ax.fill(angles, values, alpha=0.1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylim(0, 1)
ax.set_title(
    "Relative Docking & Developability Profile (display-only, in-batch)",
    fontsize=10,
)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)
plt.show()

## 12. 结果解释（固定科学措辞）

In [ ]:
result_md = (job_dir / "reports" / "result.md").read_text(encoding="utf-8")
# 只展示报告前 40 行，完整报告见 reports/result.md
print("\n".join(result_md.splitlines()[:40]))

## 13. 科学限制

- Vina affinity 是**经验打分函数与构象搜索的结果**，不能直接等同于
  实验结合自由能、Kd、Ki、IC50 或任何药效结论；
- 受体被视为**刚性**，不考虑诱导契合、蛋白柔性与显式溶剂效应；
- 质子化状态采用工具默认策略，pH 依赖效应未显式建模；
- 不同受体、不同 box、不同质子化状态或不同参数之间的分数**不可横向比较**；
- 多个 pose 分数接近时存在**构象不确定性**，最优 pose 不一定代表真实结合模式；
- 本流程输出用于**假设生成与排序**，不替代实验验证。

## 14. manifest（可复现记录）

In [ ]:
manifest = json.loads((job_dir / "manifest.json").read_text(encoding="utf-8"))
for key in (
    "job_id", "input_sha256", "receptor_sha256", "vina_version",
    "rdkit_version", "meeko_version", "python_version", "macos_version",
    "architecture", "random_seed", "status", "runtime_seconds",
):
    print(f"{key}: {manifest.get(key)}")
print("parameters:", json.dumps(manifest["parameters"]))
print("commands_or_api_calls:")
for call in manifest["commands_or_api_calls"]:
    print(" -", call)
print(f"artifact_sha256: {len(manifest['artifact_sha256'])} files hashed")

## 15. 产物列表

In [ ]:
files = sorted(p.relative_to(job_dir) for p in job_dir.rglob("*") if p.is_file())
print(f"job dir: outputs/{job_dir.name}/")
print(f"{len(files)} files:")
for rel in files:
    print(" ", rel)